In [2]:
%pip install langchain openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 2.7 MB/s  0:00:0036m-:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15/15 [langchain]15 [langgraph]sdk]]col]
Note: you may need to restart the kernel to use updated packages.


In [23]:
import langchain
import langchain_core.prompts
import langgraph
import langgraph.graph
from langchain_groq import ChatGroq
import psycopg2
import operator

import typing

import os
import json
import dotenv

dotenv.load_dotenv()


True

In [ ]:
llm_groq = ChatGroq(
    api_key=os.getenv("GROQ_API"),
    model=os.getenv("MODEL"),
    temperature=0,
)
prompt = langchain_core.prompts.ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant."),
        ("user", "Answer the following question: {text}"),
    ]
)
chain = (
    prompt
    | llm_groq
    | langchain_core.output_parsers.StrOutputParser() 
)

chain.invoke({'text': 'What is the capital of France?'})

'The capital of France is **Paris**.'

In [26]:
class State(typing.TypedDict):
    messages: typing.Annotated[list, operator.add]
    querry: str
    
def topic_prompt(state: State):
    pr = f"Answer the following question: {state['querry']}"
    return {"messages": [("system", pr)]}

def llm_for_text(state: State):
    messages = state['messages']
    response = llm_groq.invoke(messages)
    return {"messages": [("assistant", response.content)]}
graph = langgraph.graph.StateGraph(State)

graph.add_node('topic_prompt', topic_prompt)
graph.add_node('llm_for_text', llm_for_text)
graph.add_edge(langgraph.graph.START, 'topic_prompt')
graph.add_edge('topic_prompt', 'llm_for_text')
graph.add_edge('llm_for_text', langgraph.graph.END)

app = graph.compile()

app.invoke({'messages': [], 'querry': 'What is the capital of France?'})

{'messages': [('system',
   'Answer the following question: What is the capital of France?'),
  ('assistant', 'The capital of France is **Paris**.')],
 'querry': 'What is the capital of France?'}

In [13]:
generate_text("Привет, как дела?")

'Привет! У меня всё отлично, спасибо, что спросил. Как у тебя дела?'

In [27]:
conn = psycopg2.connect(
    host='localhost',
    port=os.getenv("POSTGRES_PORT"),
    database=os.getenv("POSTGRES_DB"),
    user=os.getenv("POSTGRES_USER"),
    password=os.getenv("POSTGRES_PASSWORD")
)

In [28]:
cur = conn.cursor()

In [29]:
cur.execute("SELECT * FROM messages")
for i in cur.fetchall():
    print(i[5])

Щ
Тест
Во сколько?
